# 05 - Sentiment va brand health

Notebook do ty le sentiment, phat hien ngay co negative spike va xem rieng Highlands Coffee Vietnam.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
EXPORT_DIR = ROOT / 'dashboard' / 'exports'
PROCESSED_DIR = ROOT / 'data' / 'processed'

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 140)


def read_csv(relative_path):
    path = ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f'Missing required input: {path}')
    return pd.read_csv(path)


def pct(numerator, denominator):
    return np.where(pd.Series(denominator).replace(0, np.nan).notna(), numerator / denominator * 100, np.nan)

sentiment = read_csv('dashboard/exports/vw_sentiment_trend.csv')
comments = read_csv('data/processed/comments.csv')


In [ ]:
overall = pd.DataFrame([{
    'comment_count': int(sentiment['comment_count'].sum()),
    'positive_count': int(sentiment['positive_count'].sum()),
    'neutral_count': int(sentiment['neutral_count'].sum()),
    'negative_count': int(sentiment['negative_count'].sum()),
    'positive_pct': sentiment['positive_count'].sum() / sentiment['comment_count'].sum() * 100,
    'negative_pct': sentiment['negative_count'].sum() / sentiment['comment_count'].sum() * 100,
    'avg_sentiment_score': comments['sentiment_score'].mean(),
}])
display(overall.round(4))


In [ ]:
by_page = sentiment.groupby(['page_name', 'is_competitor'], as_index=False).agg(
    comment_count=('comment_count', 'sum'),
    positive_count=('positive_count', 'sum'),
    neutral_count=('neutral_count', 'sum'),
    negative_count=('negative_count', 'sum'),
)
by_page['positive_pct'] = np.where(by_page['comment_count'] > 0, by_page['positive_count'] / by_page['comment_count'] * 100, np.nan)
by_page['negative_pct'] = np.where(by_page['comment_count'] > 0, by_page['negative_count'] / by_page['comment_count'] * 100, np.nan)
display(by_page.sort_values(['comment_count', 'positive_pct'], ascending=False).round(4))


In [ ]:
negative_spikes = sentiment[sentiment['negative_count'] > 0].sort_values(['negative_pct', 'negative_count'], ascending=False)
display(negative_spikes[['full_date', 'page_name', 'comment_count', 'negative_count', 'negative_pct', 'avg_sentiment_score']])
posts = read_csv('data/processed/posts.csv')
brand_post_ids = posts.query("page_name == 'Highlands Coffee Vietnam'")['post_id']
brand_comments = comments[comments['post_id'].isin(brand_post_ids)]
display(brand_comments.groupby('sentiment_label').size().rename('comments').reset_index())


## Insight

- Key finding: Brand health thien ve neutral, positive con thap nhung negative chua lan rong.
- Supporting data: 1,526 comments gom 351 positive, 1,148 neutral, 27 negative; positive ratio 23.00%, negative ratio 1.77%.
- Recommended action: Tang CTA goi phan hoi tich cuc trong video san pham va xu ly rieng comment negative ngay 2026-04-27 ve trai nghiem cua hang.